# MSD-H6 — Post-Selection (Level 1 + Level 2)

Magic-state-distillation (Magic-H6) benchmark. This notebook is a thin driver
over the benchmark runner in
`benchmarks/magic_h6_benchmark/run_magic_h6_benchmark.py`. It reuses that
module's sweep logic (`_run_level1` / `_run_level2`, including CSV
checkpointing) and only adds interactive configuration plus a DataFrame view.

Both levels score by **post-selection**: any fired detector / failed check
discards the shot (no error correction is applied), and `logical_error_rate`
is the residual failure rate on the surviving shots.

- Level 1: `[[6,2,2]]` detector/observable scoring
- Level 2: `[[36,4,4]]` — detector post-selection plus an explicit split of the
  tracker observables into the 4 level-2 logicals (failure) and the 4 outer
  X-stabilizers + resource/aux frame checks (post-select). LER is still an
  upper bound vs. the CQCL Magic-H6 figure of merit, but only because of the
  noise model (no separate `p_in`; resource encoders are noisy here).

In [1]:
# --- Setup ----------------------------------------------------------------
# Put the LightStim repo root (two levels up from this notebook) on sys.path,
# then load the Magic-H6 benchmark runner. It is a standalone script, not an
# installed package, so importlib loads it by file path into the name `bench`.
# Every simulation below goes through `bench._run_level1` / `bench._run_level2`
# unchanged -- this notebook only supplies config and shows the results.
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path
from types import SimpleNamespace

import pandas as pd

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

_BENCH_PATH = ROOT / "benchmarks" / "magic_h6_benchmark" / "run_magic_h6_benchmark.py"
_spec = importlib.util.spec_from_file_location("run_magic_h6_benchmark", _BENCH_PATH)
bench = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(bench)

# Same results file / checkpoint store the CLI uses by default. The runner
# appends one row per (config) here and skips configs already present, so
# re-running a cell is cheap ("SKIP ..." lines, no new rows).
OUT_PATH = ROOT / "benchmarks" / "magic_h6_benchmark" / "results" / "h6_results.csv"

## Step 0 — validate the pipeline

Before driving the benchmark, run the Magic-H6 unit suite
(`tests/test_magic_h6_benchmark.py`). It exercises every file the pipeline touches:

| Component | File | Covered by |
| --- | --- | --- |
| `[[6,2,2]]` / `[[36,4,4]]` builders | `lightstim/protocols/magic_h6_benchmark.py` | `build_h6_circuit`, `_build_h6_level2_circuit` |
| encoder + SE + H-check blocks | `lightstim/qec_code/six_two_two/` | `get_dist_circ`, `SixTwoTwoExtractionBlock`, `SixTwoTwoLogicalXCheckBlock` |
| noise injection | `lightstim/noise/…` | `inject_noise` (`full` / `idle`) |
| level-1 scoring | `lightstim/simulation/decoder_backend/…` | `run_simulation` (detector post-selection + PyMatching) |
| level-2 observable split | `lightstim/protocols/magic_h6_benchmark.py` | `_split_h6_level2_observables` (4 logicals vs. outer stabilizers) |
| level-2 scoring | `lightstim/protocols/magic_h6_benchmark.py` | `run_simulation_level2` (detector + observable post-selection) |

If any test fails, the cell raises and the sweep below is skipped — a broken
component would only produce garbage rows.

In [2]:
# --- Step 0: validate every pipeline component ---------------------------
# Shell out to pytest on tests/test_magic_h6_benchmark.py in a subprocess (this
# kernel's own Python, repo root as CWD). The tests cover the [[6,2,2]] /
# [[36,4,4]] circuit builders, the encoder / syndrome-extraction / H-check
# blocks, the noise injector, and both scoring paths. A non-zero exit code
# raises here so the sweep below never runs against a broken component.
import subprocess

_TESTS = ROOT / "tests" / "test_magic_h6_benchmark.py"
print(f"Validating pipeline via {_TESTS.relative_to(ROOT)} ...\n")

# capture_output=True so the full pytest report is shown inline in the cell.
_proc = subprocess.run(
    [sys.executable, "-m", "pytest", str(_TESTS), "-v", "--no-header", "-p", "no:cacheprovider"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(_proc.stdout)
if _proc.stderr.strip():
    print(_proc.stderr)

# Non-zero exit => at least one test failed; stop the notebook here.
if _proc.returncode != 0:
    raise RuntimeError(
        f"Magic-H6 pipeline validation FAILED (pytest exit {_proc.returncode}). "
        "Fix the failing component above before running the sweep."
    )
print("OK -- every Magic-H6 pipeline component validated.")

Validating pipeline via tests\test_magic_h6_benchmark.py ...



============================= test session starts =============================
collected 8 items

tests\test_magic_h6_benchmark.py ........                                [100%]

============================== 8 passed in 4.80s ==============================

OK -- every Magic-H6 pipeline component validated.


In [ ]:
# --- Config factory ----------------------------------------------------
# bench._run_level{1,2} were written to take an argparse.Namespace from the
# CLI. make_args() forges the same attribute bag with SimpleNamespace so the
# runner can be driven from Python. The keyword-only params ARE the sweep
# knobs:
#   mode           "full" = circuit-level depolarizing noise on every op;
#                  "idle" additionally depolarizes idling data qubits at p/5
#   p_values       level-1 physical error rates to sweep (one CSV row per
#                  value/level)
#   p_values_l2    level-2 physical error rates. The runner keeps a SEPARATE
#                  L2 grid (bench._run_level2 reads args.p_values_l2): the
#                  36-qubit patch post-selects on ~120 detectors, so
#                  acceptance collapses above ~1e-3 and L2 wants lower p than
#                  L1. At p=1e-2 L2 acceptance is ~1e-3 -- pure post-selection
#                  gets no statistics there, so keep 1e-2 OUT of this grid.
#                  Defaults to None -> reuse p_values (fine only when p_values
#                  is itself <= ~1e-3).
#   max_shots_l1   level-1 stop cap: max shots to sample
#   max_errors_l1  level-1 stop cap: stop after this many logical failures
#                  (whichever cap trips first -> why "seconds" varies wildly)
#   batch_size_l1  level-1 shots per sampling batch
#   num_samples_l2 level-2 hard cap on shots to sample (acceptance is tiny, so
#                  this needs to be large -- 1e6+ for a real point)
#   max_errors_l2  level-2 early-stop (level-1 style): stop once this many
#                  SURVIVING shots have failed, so a usable-acceptance point
#                  doesn't burn the whole num_samples_l2 budget. 0 disables it.
#   batch_size_l2  level-2 shots per sampling batch -- also the granularity of
#                  the max_errors_l2 early-stop (it can only trip on a batch
#                  boundary). Not a result knob.
#   num_workers    KEEP AT 1 on Windows/Jupyter -- see docstring below
#   print_progress stream the runner's per-batch progress lines
#
# There is no `rounds` knob: build_h6_circuit hardcodes the level-1 [[6,2,2]]
# block to exactly one syndrome-extraction round, and level 2 has no rounds.
def make_args(
    *,
    mode: str = "full",
    p_values: tuple[float, ...] = (1e-2,),
    p_values_l2: tuple[float, ...] | None = None,
    max_shots_l1: int = 20_000,
    max_errors_l1: int = 20,
    batch_size_l1: int = 5_000,
    num_samples_l2: int = 50_000,
    max_errors_l2: int = 20,
    batch_size_l2: int = 10_000,
    num_workers: int = 1,
    print_progress: bool = False,
) -> SimpleNamespace:
    """Build the argument namespace that run_magic_h6_benchmark._run_level{1,2} expect.

    ``num_workers`` defaults to 1 so the level-1 pipeline uses its
    single-process path. The multi-process path relies on ``multiprocessing``
    with the "spawn" start method (Windows / Jupyter), where worker processes
    cannot re-import the notebook and the run silently yields ``shots=0``.
    Bump it to a higher value only when running this notebook from a
    Linux/macOS kernel where "fork" is available.

    ``p_values_l2`` is the level-2 sweep grid, kept separate because the
    runner's ``_run_level2`` iterates ``args.p_values_l2``. Pass ``None`` to
    reuse ``p_values`` for level 2 as well -- but only do that when
    ``p_values`` is already <= ~1e-3, since L2 acceptance collapses above that.

    ``num_samples_l2`` is a hard shot cap and ``max_errors_l2`` is the
    level-1-style early-stop (stop once that many surviving shots failed,
    checked at each ``batch_size_l2`` boundary); whichever trips first ends the
    L2 point. Set ``max_errors_l2=0`` to sample the full ``num_samples_l2``
    every time.
    """
    return SimpleNamespace(
        mode=mode,
        p_values=list(p_values),
        p_values_l2=list(p_values if p_values_l2 is None else p_values_l2),
        max_shots_l1=max_shots_l1,
        max_errors_l1=max_errors_l1,
        batch_size_l1=batch_size_l1,
        num_samples_l2=num_samples_l2,
        max_errors_l2=max_errors_l2,
        batch_size_l2=batch_size_l2,
        num_workers=num_workers,
        print_progress=print_progress,
    )


# Default config: level 1 at p = 1e-2, level 2 at p = 1e-3 (a point where L2
# post-selection still accepts enough shots to score). Small budgets = fast
# smoke run.
args = make_args(p_values_l2=(1e-3,))
args


In [4]:
# --- Driver ----------------------------------------------------------
# Thin wrapper over the runner's own sweep loops. bench._run_level1 /
# bench._run_level2 build the circuit, sweep args.p_values, and append one row
# per (config) to `out_path`, skipping any (config) already in the file. Then
# we reload the whole CSV as a DataFrame.
#
#   * Re-running with a config already in the CSV prints "SKIP ..." and adds
#     nothing.
#   * The returned frame is the ENTIRE file (every config ever run), sorted by
#     level then p -- not just this call's rows. Filter with the mask cell
#     below to see only `args`.
def run_h6(args: SimpleNamespace, levels=(1, 2), out_path: Path = OUT_PATH) -> pd.DataFrame:
    """Drive the benchmark runner for the requested levels, then return its CSV."""
    if 1 in levels:
        bench._run_level1(args, out_path)
    if 2 in levels:
        bench._run_level2(args, out_path)

    df = pd.read_csv(out_path)
    return df.sort_values(["level", "p"]).reset_index(drop=True)

In [ ]:
# --- Quick run: both levels, default `args` (L1 @ p=0.01, L2 @ p=0.001) --
# First execution simulates and appends to the CSV; reruns print "SKIP ..."
# and add nothing.
#
# NOTE: `df` is the WHOLE results file sorted by level then p -- it includes
# any rows left from earlier sweeps, not just this `args`. The next cell
# filters down to the current config.
df = run_h6(args, levels=(1, 2))
df


In [ ]:
# --- Where the numbers live + focus on the current config --------------
# Nothing to save here -- the runner already persisted every row to this CSV.
print(OUT_PATH)

# Filter the full table down to just what THIS run's `args` selected:
# same noise mode AND p in either sweep grid (L1 uses args.p_values, L2 uses
# args.p_values_l2 -- they differ now that 1e-2 is kept off the L2 grid).
# You may still get >1 row per level here if the CSV holds the same p under
# different shot budgets (max_shots_l1 / num_samples_l2 are part of the key).
_p_current = set(args.p_values) | set(args.p_values_l2)
mask = (df["mode"] == args.mode) & (df["p"].isin(_p_current))
df[mask]


In [ ]:
# --- Custom sweep: same runner, wider p grid + bigger shot budget ------
# Template for a real data run. Builds a fresh `args` (does NOT reuse the
# default one above):
#   * level 1 over p = 3e-3 / 5e-3 / 1e-2, 50k-shot cap, 50 errors
#   * level 2 over p = 3e-3 / 5e-3 ONLY -- 1e-2 is dropped because L2
#     acceptance there is ~1e-3 and pure post-selection never accumulates
#     statistics. L2 samples up to 500k shots but early-stops at 100 failures,
#     so the low-acceptance points don't run forever.
# New (config) rows are appended to the same CSV; any point already computed
# is skipped ("SKIP ..."). Raise num_samples_l2 / max_errors_l2 for tighter
# level-2 error bars.
args_sweep = make_args(
    mode="full",
    p_values=(3e-3, 5e-3, 1e-2),
    p_values_l2=(3e-3, 5e-3),
    max_shots_l1=50_000,
    max_errors_l1=50,
    batch_size_l1=10_000,
    num_samples_l2=500_000,
    max_errors_l2=100,
)

# df_sweep = the full CSV again (all levels, all p), including any rows this
# sweep just added.
df_sweep = run_h6(args_sweep, levels=(1, 2))
df_sweep
